# Reasoning Models — Foundations

> **Section 01 — Topic 08 — foundations**

**Prerequisites:** Understanding of LLM inference (Topic 07)

**What you'll learn:**
- Chain-of-thought prompting and why it works
- Inference-time compute scaling: more thinking = better answers
- Self-consistency through multiple reasoning chains
- Step-by-step decomposition strategies

**What you'll build:**
- Complete implementations of CoT, self-consistency, and plan-and-solve
- Side-by-side accuracy comparisons with visualizations

## Setup

We use the OpenAI API for most experiments (easy to swap for any provider). The reasoning patterns we implement work with any LLM.

In [ ]:
!pip install -q openai anthropic matplotlib numpy

In [ ]:
import os
import re
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from openai import OpenAI

# Initialize client — set your API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-key-here"))

# Model to use — gpt-4o-mini is fast and cheap for experimentation
MODEL = "gpt-4o-mini"

def chat(messages, model=MODEL, temperature=0.0, max_tokens=1024):
    """Simple wrapper for chat completions."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

def ask(prompt, **kwargs):
    """Quick single-turn query."""
    return chat([{"role": "user", "content": prompt}], **kwargs)

print(f"Using model: {MODEL}")

---
## 1. Chain-of-Thought Prompting

Chain-of-thought (CoT) prompting (Wei et al., 2022) is the single most important reasoning technique. The insight is simple but profound: **asking a model to show its work dramatically improves accuracy on reasoning tasks.**

Why does this work? LLMs generate tokens sequentially. Each token can only attend to previous tokens. By generating intermediate reasoning steps, the model creates a "scratchpad" that subsequent tokens can attend to. Without CoT, the model must jump directly from question to answer in a single token-prediction step.

### 1.1 Zero-Shot CoT

The simplest version: just add "Let's think step by step" to the prompt. This was shown by Kojima et al. (2022) to dramatically improve performance with zero examples.

In [ ]:
# Test problems — mix of math, logic, and common sense
problems = [
    {
        "question": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        "answer": "0.05",  # $0.05 — the classic cognitive reflection test
    },
    {
        "question": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
        "answer": "5",  # 5 minutes — each machine makes 1 widget in 5 minutes
    },
    {
        "question": "A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?",
        "answer": "9",
    },
    {
        "question": "If you have a 3-gallon jug and a 5-gallon jug, how can you measure exactly 4 gallons of water? Just give the final amount.",
        "answer": "4",
    },
    {
        "question": "What is 17 * 23 + 45 - 12 * 3?",
        "answer": "400",  # 391 + 45 - 36 = 400
    },
]

def extract_number(text):
    """Extract the final number from a response."""
    # Look for numbers in the last line or after common patterns
    numbers = re.findall(r'[-+]?\d*\.?\d+', text.split('\n')[-1])
    if numbers:
        return numbers[-1]
    numbers = re.findall(r'[-+]?\d*\.?\d+', text)
    return numbers[-1] if numbers else None

# Direct prompting (no CoT)
print("=" * 70)
print("DIRECT PROMPTING (no chain-of-thought)")
print("=" * 70)
direct_correct = 0

for p in problems:
    response = ask(f"{p['question']}\nAnswer with just the number.")
    extracted = extract_number(response)
    correct = extracted == p['answer'] if extracted else False
    direct_correct += int(correct)
    status = "CORRECT" if correct else "WRONG"
    print(f"\n  Q: {p['question'][:60]}...")
    print(f"  A: {response.strip()[:80]} [{status}]")

print(f"\nDirect accuracy: {direct_correct}/{len(problems)}")

In [ ]:
# Zero-shot CoT
print("=" * 70)
print("ZERO-SHOT CHAIN-OF-THOUGHT")
print("=" * 70)
cot_correct = 0

for p in problems:
    prompt = f"{p['question']}\n\nLet's think step by step."
    response = ask(prompt)
    extracted = extract_number(response)
    correct = extracted == p['answer'] if extracted else False
    cot_correct += int(correct)
    status = "CORRECT" if correct else "WRONG"
    print(f"\n  Q: {p['question'][:60]}...")
    print(f"  Reasoning: {response.strip()[:150]}...")
    print(f"  Answer: {extracted} [{status}]")

print(f"\nCoT accuracy: {cot_correct}/{len(problems)}")

### 1.2 Few-Shot CoT

Even better: provide examples of step-by-step reasoning. The model learns the *format* of careful reasoning from the examples.

In [ ]:
# Few-shot CoT with demonstration examples
few_shot_examples = """I'll show you how to solve problems step by step.

Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: Let's think step by step.
1. Roger starts with 5 tennis balls.
2. He buys 2 cans, each with 3 balls: 2 * 3 = 6 new balls.
3. Total: 5 + 6 = 11 tennis balls.
The answer is 11.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, then 1 leaves, how many cars are in the parking lot?
A: Let's think step by step.
1. Start with 3 cars.
2. 2 more arrive: 3 + 2 = 5 cars.
3. 1 leaves: 5 - 1 = 4 cars.
The answer is 4."""

print("=" * 70)
print("FEW-SHOT CHAIN-OF-THOUGHT")
print("=" * 70)
fewshot_correct = 0

for p in problems:
    prompt = f"{few_shot_examples}\n\nQ: {p['question']}\nA: Let's think step by step."
    response = ask(prompt)
    extracted = extract_number(response)
    correct = extracted == p['answer'] if extracted else False
    fewshot_correct += int(correct)
    status = "CORRECT" if correct else "WRONG"
    print(f"\n  Q: {p['question'][:60]}...")
    print(f"  Answer: {extracted} [{status}]")

print(f"\nFew-shot CoT accuracy: {fewshot_correct}/{len(problems)}")

---
## 2. Inference-Time Compute Scaling

A fundamental insight from recent research: **you can trade more compute at inference time for better answers.** This is the core idea behind reasoning models like o1 and o3.

There are multiple ways to spend more inference-time compute:
1. **Longer chains of thought** — more reasoning tokens
2. **Multiple samples** — generate several answers and pick the best
3. **Iterative refinement** — critique and improve answers
4. **Search** — tree of thought, MCTS

Let's measure how answer quality scales with compute.

In [ ]:
# Experiment: How does answer quality scale with number of reasoning tokens?

math_problems = [
    {"q": "What is 15% of 240?", "a": "36"},
    {"q": "A train travels at 60 mph for 2.5 hours. How far does it go?", "a": "150"},
    {"q": "If 3x + 7 = 22, what is x?", "a": "5"},
    {"q": "What is the sum of the first 10 positive integers?", "a": "55"},
    {"q": "A rectangle has area 48 and width 6. What is its perimeter?", "a": "28"},
    {"q": "What is 2^10?", "a": "1024"},
    {"q": "If a shirt costs $25 after a 20% discount, what was the original price?", "a": "31.25"},
    {"q": "How many seconds are in 3 hours and 45 minutes?", "a": "13500"},
]

def test_with_token_budget(problems, max_tokens_list):
    """Test accuracy across different reasoning budgets."""
    results = {}
    
    for max_t in max_tokens_list:
        correct = 0
        for p in problems:
            response = ask(
                f"{p['q']}\n\nThink step by step, then give the final numerical answer.",
                max_tokens=max_t
            )
            extracted = extract_number(response)
            if extracted == p['a']:
                correct += 1
        
        accuracy = correct / len(problems)
        results[max_t] = accuracy
        print(f"  max_tokens={max_t:4d}: accuracy={accuracy:.1%} ({correct}/{len(problems)})")
    
    return results

print("Accuracy vs reasoning budget:")
budget_results = test_with_token_budget(math_problems, [32, 64, 128, 256, 512])

In [ ]:
# Visualize inference-time compute scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs token budget
tokens = list(budget_results.keys())
accs = list(budget_results.values())

ax1.plot(tokens, accs, 'o-', color='#2ecc71', linewidth=2, markersize=8)
ax1.set_xlabel('Max tokens (reasoning budget)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy Scales with Reasoning Budget')
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

# Conceptual: inference-time scaling law
compute_levels = np.linspace(1, 100, 100)
# Approximate log-linear scaling (observed empirically)
performance = 0.3 + 0.15 * np.log(compute_levels)
performance = np.clip(performance, 0, 1)

ax2.plot(compute_levels, performance, color='#3498db', linewidth=2)
ax2.fill_between(compute_levels, performance - 0.05, performance + 0.05, alpha=0.2, color='#3498db')
ax2.set_xlabel('Relative inference-time compute')
ax2.set_ylabel('Task performance')
ax2.set_title('Inference-Time Compute Scaling Law (Conceptual)')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)
ax2.annotate('Diminishing returns', xy=(60, 0.85), fontsize=10, style='italic', color='gray')

plt.tight_layout()
plt.show()

print("Key insight: performance scales log-linearly with inference-time compute.")
print("This means doubling compute gives a fixed accuracy improvement — not linear.")

---
## 3. Self-Consistency — Multiple Chains + Majority Vote

Self-consistency (Wang et al., 2023) is one of the most effective reasoning techniques. The idea:

1. Generate $N$ independent reasoning chains (with temperature > 0)
2. Extract the final answer from each chain
3. Take the **majority vote** as the final answer

This works because correct reasoning paths tend to converge on the right answer, while incorrect paths scatter across different wrong answers.

**Analogy:** If you ask 10 people to solve a math problem and 7 get the same answer, that answer is probably right — even if 3 people made different mistakes.

In [ ]:
def self_consistency(question, n_chains=5, temperature=0.7):
    """
    Self-consistency: generate N reasoning chains and majority-vote.
    """
    prompt = f"{question}\n\nLet's think step by step."
    
    chains = []
    answers = []
    
    for i in range(n_chains):
        response = ask(prompt, temperature=temperature, max_tokens=512)
        chains.append(response)
        
        # Extract numerical answer
        extracted = extract_number(response)
        if extracted:
            answers.append(extracted)
    
    if not answers:
        return None, chains, answers, {}
    
    # Majority vote
    vote_counts = Counter(answers)
    majority_answer = vote_counts.most_common(1)[0][0]
    
    return majority_answer, chains, answers, dict(vote_counts)

# Test self-consistency on a tricky problem
question = "A store sells apples for $0.50 each. If you buy 5 or more, you get a 20% discount on the total. How much do 7 apples cost?"
expected = "2.80"

print(f"Question: {question}")
print(f"Expected: ${expected}")
print()

answer, chains, all_answers, votes = self_consistency(question, n_chains=7)

print(f"Individual chain answers: {all_answers}")
print(f"Vote distribution: {votes}")
print(f"Majority vote answer: {answer}")
print(f"Correct: {answer == expected}")

In [ ]:
# Systematic comparison: single CoT vs self-consistency with increasing N

test_problems = [
    {"q": "What is 17 * 23 + 45 - 12 * 3?", "a": "400"},
    {"q": "A bat and ball cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost in dollars?", "a": "0.05"},
    {"q": "If 8 workers can build a wall in 10 hours, how many hours would 5 workers take?", "a": "16"},
    {"q": "What is the remainder when 2847 is divided by 13?", "a": "0"},
    {"q": "A car depreciates 15% per year. After 2 years, what fraction of the original price remains? Give as a decimal.", "a": "0.7225"},
]

n_values = [1, 3, 5, 9, 15]
sc_results = {}

print("Self-consistency accuracy vs number of chains:\n")
for n in n_values:
    correct = 0
    for p in test_problems:
        if n == 1:
            # Single chain = regular CoT
            response = ask(f"{p['q']}\n\nLet's think step by step.", temperature=0.0)
            extracted = extract_number(response)
            if extracted == p['a']:
                correct += 1
        else:
            answer, _, _, _ = self_consistency(p['q'], n_chains=n)
            if answer == p['a']:
                correct += 1
    
    accuracy = correct / len(test_problems)
    sc_results[n] = accuracy
    print(f"  N={n:2d} chains: accuracy={accuracy:.1%} ({correct}/{len(test_problems)})")

In [ ]:
# Visualize self-consistency scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ns = list(sc_results.keys())
accs = list(sc_results.values())

ax1.plot(ns, accs, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax1.set_xlabel('Number of reasoning chains (N)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Self-Consistency: More Chains = Better Accuracy')
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

# Show vote distribution example
example_votes = votes  # From the previous cell
if example_votes:
    answers_sorted = sorted(example_votes.keys())
    counts = [example_votes[a] for a in answers_sorted]
    bars = ax2.bar(answers_sorted, counts, color='#3498db', edgecolor='white')
    # Highlight majority
    max_count = max(counts)
    for bar, count in zip(bars, counts):
        if count == max_count:
            bar.set_color('#2ecc71')
    ax2.set_xlabel('Answer')
    ax2.set_ylabel('Vote count')
    ax2.set_title('Self-Consistency Vote Distribution')

plt.tight_layout()
plt.show()

---
## 4. Step-by-Step Reasoning — Decomposition Strategies

Beyond basic CoT, there are more structured ways to decompose problems.

### 4.1 Least-to-Most Prompting

Zhou et al. (2023): Break a complex problem into simpler sub-problems, solve each in order, building up to the final answer. Unlike CoT which writes out reasoning in one pass, least-to-most *explicitly decomposes* first.

### 4.2 Plan-and-Solve

Wang et al. (2023): First generate a plan, then execute each step. This gives the model a roadmap before it starts working.

In [ ]:
def least_to_most(question):
    """
    Least-to-Most prompting:
    1. Decompose the problem into sub-problems
    2. Solve each sub-problem in order
    3. Combine for final answer
    """
    # Step 1: Decompose
    decompose_prompt = f"""Break down this problem into simpler sub-problems that build on each other.
List them from simplest to most complex.

Problem: {question}

Sub-problems (from simplest to hardest):"""
    
    decomposition = ask(decompose_prompt)
    print(f"  Decomposition:\n{decomposition}\n")
    
    # Step 2: Solve progressively
    solve_prompt = f"""Here is a problem broken into sub-problems. Solve each one in order,
using the answers from previous sub-problems.

Original problem: {question}

Sub-problems:
{decomposition}

Solve each sub-problem step by step, then give the final answer to the original problem."""
    
    solution = ask(solve_prompt)
    return solution


def plan_and_solve(question):
    """
    Plan-and-Solve prompting:
    1. Create a plan
    2. Execute the plan step by step
    """
    prompt = f"""Let's solve this problem with a plan.

Problem: {question}

First, let's devise a plan to solve this:
Plan:
1. Identify the relevant variables and what we need to find.
2. Determine the mathematical relationships.
3. Calculate step by step.
4. Verify the answer.

Now let's carry out this plan:"""
    
    return ask(prompt)

# Test on a complex word problem
complex_problem = (
    "A store offers a 'buy 3, get 1 free' deal on shirts that cost $45 each. "
    "If you also have a 10% off coupon applied after the deal, "
    "how much would 8 shirts cost?"
)

print("LEAST-TO-MOST:")
print("=" * 50)
l2m_answer = least_to_most(complex_problem)
print(f"Solution: {l2m_answer}")

In [ ]:
print("\nPLAN-AND-SOLVE:")
print("=" * 50)
ps_answer = plan_and_solve(complex_problem)
print(f"Solution: {ps_answer}")

In [ ]:
# Compare all approaches on a standard problem
print("\nDIRECT (no reasoning):")
direct = ask(f"{complex_problem}\nJust give the final dollar amount.")
print(f"  {direct}")

print("\nZERO-SHOT CoT:")
cot = ask(f"{complex_problem}\nLet's think step by step.")
print(f"  {cot[:200]}...")

---
## 5. Comparing Approaches — Accuracy vs Compute

Let's do a systematic comparison across multiple problems and strategies.

In [ ]:
# Benchmark suite
benchmark = [
    {"q": "What is 15% of 240?", "a": "36"},
    {"q": "If 3x + 7 = 22, what is x?", "a": "5"},
    {"q": "A rectangle has perimeter 30 and length 10. What is its area?", "a": "50"},
    {"q": "What is the sum of the first 10 positive integers?", "a": "55"},
    {"q": "A car travels 180 miles in 3 hours. What is its average speed in mph?", "a": "60"},
    {"q": "If you invest $1000 at 5% simple interest for 3 years, what is the total interest earned?", "a": "150"},
    {"q": "How many ways can you arrange 4 books on a shelf?", "a": "24"},
    {"q": "What is the GCD of 48 and 36?", "a": "12"},
]

strategies = {
    "Direct": lambda q: ask(f"{q}\nAnswer with just the number."),
    "Zero-shot CoT": lambda q: ask(f"{q}\nLet's think step by step."),
    "Plan-and-Solve": lambda q: plan_and_solve(q),
}

all_results = {}
all_times = {}

for name, strategy in strategies.items():
    correct = 0
    start = time.time()
    
    for p in benchmark:
        response = strategy(p['q'])
        extracted = extract_number(response)
        if extracted == p['a']:
            correct += 1
    
    elapsed = time.time() - start
    accuracy = correct / len(benchmark)
    all_results[name] = accuracy
    all_times[name] = elapsed
    print(f"{name:20s}: {accuracy:.1%} accuracy, {elapsed:.1f}s total")

# Also test self-consistency (5 chains)
sc_correct = 0
start = time.time()
for p in benchmark:
    answer, _, _, _ = self_consistency(p['q'], n_chains=5)
    if answer == p['a']:
        sc_correct += 1
sc_elapsed = time.time() - start
sc_acc = sc_correct / len(benchmark)
all_results["Self-Consistency (N=5)"] = sc_acc
all_times["Self-Consistency (N=5)"] = sc_elapsed
print(f"{'Self-Consistency (N=5)':20s}: {sc_acc:.1%} accuracy, {sc_elapsed:.1f}s total")

In [ ]:
# Final comparison visualization
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

names = list(all_results.keys())
accuracies = list(all_results.values())
times = list(all_times.values())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

# Accuracy
bars = ax1.bar(range(len(names)), accuracies, color=colors[:len(names)], edgecolor='white')
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy by Strategy')
ax1.set_ylim(0, 1.1)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.0%}', ha='center', fontweight='bold', fontsize=10)

# Time
ax2.bar(range(len(names)), times, color=colors[:len(names)], edgecolor='white')
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Compute Cost by Strategy')

# Efficiency: accuracy per second
efficiency = [a / t if t > 0 else 0 for a, t in zip(accuracies, times)]
ax3.bar(range(len(names)), efficiency, color=colors[:len(names)], edgecolor='white')
ax3.set_xticks(range(len(names)))
ax3.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax3.set_ylabel('Accuracy / Second')
ax3.set_title('Efficiency (Accuracy per Unit Compute)')

fig.suptitle('Reasoning Strategy Comparison', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey takeaways:")
print("1. CoT consistently outperforms direct prompting")
print("2. Self-consistency trades compute for accuracy — worth it for hard problems")
print("3. Plan-and-solve helps on complex multi-step problems")
print("4. The best strategy depends on your accuracy vs latency requirements")

---
## Summary

| Technique | How It Works | When to Use |
|-----------|-------------|-------------|
| **Zero-shot CoT** | "Let's think step by step" | Quick, always-on improvement |
| **Few-shot CoT** | Examples with reasoning | When you have good demonstrations |
| **Self-consistency** | N chains + majority vote | When accuracy matters more than latency |
| **Least-to-most** | Decompose then solve | Complex multi-part problems |
| **Plan-and-solve** | Make plan then execute | Structured problem solving |

**The fundamental tradeoff:** more inference-time compute = better answers, with diminishing returns. The art is knowing how much compute to spend for each problem.

**Next:** `08-reasoning-models/advanced.ipynb` — o1-style reasoning, thinking tokens, tree of thought